---
## Step 0 — Mount Google Drive & Add Shared Folder

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import os, glob

# ── Auto-discover the dataset ──────────────────────────────────────────
# We search common locations for folders matching tomato class names.
# If auto-detection fails, set DATASET_ROOT manually below.

DATASET_ROOT = None
# DATASET_ROOT = "/content/drive/MyDrive/PlantVillage"

TARGET_CLASSES = [
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Septoria_leaf_spot",
    "Tomato___healthy",
]

def find_dataset_root(base="/content/drive/MyDrive"):
    """Walk Drive looking for a folder that contains our 5 tomato class subfolders."""
    # First, try to find folders with tomato class names using find
    print("🔍 Searching for tomato dataset in your Drive (this may take a minute)...")

    # Quick checks: common paths
    quick_paths = [
        os.path.join(base, "PlantVillage"),
        os.path.join(base, "Tomato_Caption", "raw", "PlantVillage"),
        os.path.join(base, "Tomato_Caption", "raw"),
        os.path.join(base, "Tomato_Caption"),
        os.path.join(base, "tomato"),
        os.path.join(base, "dataset"),
        os.path.join(base, "data"),
    ]
    for p in quick_paths:
        if os.path.isdir(p):
            subs = os.listdir(p)
            matches = sum(1 for c in TARGET_CLASSES if any(c.lower() in s.lower() for s in subs))
            if matches >= 3:
                print(f"  ✅ Found at: {p}")
                return p

    # Deep search: walk up to 3 levels deep
    for root, dirs, files in os.walk(base):
        depth = root.replace(base, "").count(os.sep)
        if depth > 3:
            dirs.clear()
            continue
        matches = sum(1 for c in TARGET_CLASSES if any(c.lower() in d.lower() for d in dirs))
        if matches >= 3:
            print(f"  ✅ Found at: {root}")
            return root

    return None

if DATASET_ROOT is None:
    DATASET_ROOT = find_dataset_root()

if DATASET_ROOT is None:
    print("\n❌ Could not auto-detect dataset!")
    print("Please set DATASET_ROOT manually in this cell and re-run.")
    print("\nListing top-level Drive contents for reference:")
    for item in sorted(os.listdir("/content/drive/MyDrive")):
        print(f"  📁 {item}" if os.path.isdir(os.path.join("/content/drive/MyDrive", item)) else f"  📄 {item}")
else:
    print(f"\n📂 DATASET_ROOT = {DATASET_ROOT}")
    print("\nContents:")
    for item in sorted(os.listdir(DATASET_ROOT)):
        full = os.path.join(DATASET_ROOT, item)
        if os.path.isdir(full):
            n_files = len([f for f in os.listdir(full) if f.lower().endswith((".jpg",".jpeg",".png"))])
            print(f"  📁 {item}/ — {n_files} images")
        else:
            print(f"  📄 {item}")

🔍 Searching for tomato dataset in your Drive (this may take a minute)...
  ✅ Found at: /content/drive/MyDrive/Tomato_Caption

📂 DATASET_ROOT = /content/drive/MyDrive/Tomato_Caption

Contents:
  📁 Tomato___Bacterial_spot/ — 331 images
  📁 Tomato___Early_blight/ — 330 images
  📁 Tomato___Late_blight/ — 329 images
  📁 Tomato___Septoria_leaf_spot/ — 302 images
  📁 Tomato___healthy/ — 330 images


---
## Step 1 — Install Dependencies

In [3]:
!pip install -q diffusers peft transformers accelerate safetensors xformers openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 52.4 MB/s eta 0:00:00


In [4]:
!pip install torchao>=0.16.0 --force-reinstall

---
## Step 2 — Imports & Device Check

In [5]:
import os, gc, csv, warnings, shutil, zipfile, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from tqdm import tqdm
from scipy import linalg
from scipy.stats import entropy
from sklearn.neighbors import NearestNeighbors

from diffusers import StableDiffusionPipeline, DDPMScheduler, DPMSolverMultistepScheduler
from transformers import CLIPTokenizer, CLIPTextModel
from diffusers import AutoencoderKL, UNet2DConditionModel
from peft import LoraConfig, get_peft_model, PeftModel

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
assert DEVICE.type == "cuda", "❌ GPU required! Go to Runtime → Change runtime type → GPU"

Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


---
## Step 3 — Configuration & Directory Setup

In [6]:
# ── Output root (everything goes here on your Drive) ─────────────────
OUTPUT_ROOT = "/content/drive/MyDrive/Tomato_SD21_Output"

# ── Paths ─────────────────────────────────────────────────────────────
LORA_DIR  = os.path.join(OUTPUT_ROOT, "sd21_lora")      # LoRA checkpoints
AUG_ROOT  = os.path.join(OUTPUT_ROOT, "augmented_v2")    # generated images
FID_REAL  = os.path.join(OUTPUT_ROOT, "fid_real_v2")     # real images (copied for eval)
FID_FAKE  = os.path.join(OUTPUT_ROOT, "fid_fake_v2")     # fake images (for eval)
SD_CACHE  = os.path.join(OUTPUT_ROOT, "sd21_cache")      # model weights cache
ZIP_PATH  = os.path.join(OUTPUT_ROOT, "augmented_v2_all_classes.zip")

CLASSES = [
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Septoria_leaf_spot",
    "Tomato___healthy",
]

VALID_EXT = (".jpg", ".jpeg", ".png")
TARGET_PER_CLASS = 2671

# ── Create all directories ────────────────────────────────────────────
for d in [OUTPUT_ROOT, LORA_DIR, AUG_ROOT, FID_REAL, FID_FAKE, SD_CACHE]:
    os.makedirs(d, exist_ok=True)
for cls in CLASSES:
    os.makedirs(os.path.join(AUG_ROOT, cls), exist_ok=True)
    os.makedirs(os.path.join(FID_REAL, cls), exist_ok=True)
    os.makedirs(os.path.join(FID_FAKE, cls), exist_ok=True)

# ── Resolve class folder mapping ─────────────────────────────────────
# The dataset folder might have slightly different naming. We map each
# of our 5 CLASSES to the actual folder name in DATASET_ROOT.
CLASS_FOLDER_MAP = {}  # CLASSES[i] -> actual folder path with images

available_dirs = [d for d in os.listdir(DATASET_ROOT) if os.path.isdir(os.path.join(DATASET_ROOT, d))]

for cls in CLASSES:
    matched = None
    # Exact match first
    if cls in available_dirs:
        matched = cls
    else:
        # Fuzzy match (case-insensitive substring)
        for d in available_dirs:
            if cls.lower() in d.lower() or d.lower() in cls.lower():
                matched = d
                break
    if matched:
        folder_path = os.path.join(DATASET_ROOT, matched)
        n_imgs = len([f for f in os.listdir(folder_path) if f.lower().endswith(VALID_EXT)])
        CLASS_FOLDER_MAP[cls] = folder_path
        print(f"  ✅ {cls:40s} → {matched}/ ({n_imgs} images)")
    else:
        print(f"  ❌ {cls:40s} → NOT FOUND!")

assert len(CLASS_FOLDER_MAP) == len(CLASSES), (
    f"Only found {len(CLASS_FOLDER_MAP)}/{len(CLASSES)} classes. "
    f"Check DATASET_ROOT and folder names."
)
print(f"\n✅ All {len(CLASSES)} classes found. Output will go to: {OUTPUT_ROOT}")

  ✅ Tomato___Bacterial_spot                  → Tomato___Bacterial_spot/ (331 images)
  ✅ Tomato___Early_blight                    → Tomato___Early_blight/ (330 images)
  ✅ Tomato___Late_blight                     → Tomato___Late_blight/ (329 images)
  ✅ Tomato___Septoria_leaf_spot              → Tomato___Septoria_leaf_spot/ (302 images)
  ✅ Tomato___healthy                         → Tomato___healthy/ (330 images)

✅ All 5 classes found. Output will go to: /content/drive/MyDrive/Tomato_SD21_Output


---
## Step 4 — LoRA Fine-Tuning Setup

Exact hyperparameters from the original pipeline:
- LoRA: r=16, α=32, dropout=0.05
- AdamW: lr=1e-4, weight_decay=1e-2
- 5 epochs per class, batch_size=2

In [7]:
SD21_MODEL_ID = "Manojb/stable-diffusion-2-1-base"

DISEASE_PROMPTS = {
    "Tomato___Early_blight":
        "macro photo tomato leaf early blight dark brown concentric ring lesions "
        "target pattern alternaria solani infection natural lighting high resolution",
    "Tomato___Late_blight":
        "macro photo tomato leaf late blight dark water-soaked irregular necrotic patches "
        "phytophthora infestans spreading dark lesions natural lighting high resolution",
    "Tomato___Septoria_leaf_spot":
        "macro photo tomato leaf septoria leaf spot numerous small circular brown spots "
        "grey centres dark borders septoria lycopersici natural lighting high resolution",
    "Tomato___Bacterial_spot":
        "macro photo tomato leaf bacterial spot small angular dark water-soaked lesions "
        "xanthomonas infection scattered spots natural lighting high resolution",
    "Tomato___healthy":
        "macro photo healthy green tomato leaf no disease uniform bright green "
        "intact margins smooth surface natural lighting high resolution",
}

NEGATIVE_PROMPT = (
    "blurry low quality out of focus dark watermark text logo "
    "cartoon illustration distorted noisy grainy"
)


class DiseaseSD21Dataset(Dataset):
    """Loads real images for a single class for LoRA fine-tuning."""
    def __init__(self, cls, tokenizer):
        self.prompt    = DISEASE_PROMPTS[cls]
        self.tokenizer = tokenizer
        self.transform = transforms.Compose([
            transforms.Resize((512, 512),
                interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.RandomHorizontalFlip(0.5),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3),
        ])
        self.img_dir = CLASS_FOLDER_MAP[cls]
        self.imgs = [os.path.join(self.img_dir, f)
                     for f in os.listdir(self.img_dir)
                     if f.lower().endswith(VALID_EXT)]
        print(f"  {cls.replace('Tomato___',''):30s}: {len(self.imgs)} real images")

    def __len__(self): return len(self.imgs)

    def __getitem__(self, idx):
        pv  = self.transform(Image.open(self.imgs[idx]).convert("RGB"))
        ids = self.tokenizer(
            self.prompt, padding="max_length",
            max_length=self.tokenizer.model_max_length,
            truncation=True, return_tensors="pt",
        ).input_ids.squeeze(0)
        return {"pixel_values": pv, "input_ids": ids}


def train_sd21_lora(cls, tokenizer, text_encoder, vae, unet, scheduler):
    """Fine-tune LoRA adapters on UNet + text encoder for one class."""
    print(f"\n{'='*60}")
    print(f"LoRA fine-tuning: {cls.replace('Tomato___','')}")
    print(f"{'='*60}")
    ds  = DiseaseSD21Dataset(cls, tokenizer)
    dl  = DataLoader(ds, batch_size=2, shuffle=True, num_workers=2)

    unet_lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["to_q","to_k","to_v","to_out.0",
                         "proj_in","proj_out","ff.net.0.proj","ff.net.2"],
        bias="none")
    te_lora   = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj","k_proj","v_proj","out_proj"], bias="none")

    unet_l    = get_peft_model(unet,         unet_lora)
    te_l      = get_peft_model(text_encoder, te_lora)
    params    = (list(filter(lambda p: p.requires_grad, unet_l.parameters())) +
                 list(filter(lambda p: p.requires_grad, te_l.parameters())))
    opt       = torch.optim.AdamW(params, lr=1e-4, weight_decay=1e-2)
    scaler    = torch.cuda.amp.GradScaler()

    unet_l.train(); te_l.train()
    for ep in range(5):
        tot = 0.0
        for batch in tqdm(dl, desc=f"  Ep{ep+1}/5", leave=False):
            pv  = batch["pixel_values"].to(DEVICE, dtype=torch.float16)
            ids = batch["input_ids"].to(DEVICE)
            with torch.cuda.amp.autocast():
                with torch.no_grad():
                    lat = vae.encode(pv).latent_dist.sample()
                    lat = lat * vae.config.scaling_factor
                noise = torch.randn_like(lat)
                t     = torch.randint(0, scheduler.config.num_train_timesteps,
                                      (lat.shape[0],), device=DEVICE).long()
                nlt   = scheduler.add_noise(lat, noise, t)
                ehs   = te_l(ids)[0]
                pred  = unet_l(nlt, t, encoder_hidden_states=ehs).sample
                tgt   = noise if scheduler.config.prediction_type=="epsilon" \
                        else scheduler.get_velocity(lat, noise, t)
                loss  = F.mse_loss(pred.float(), tgt.float())
            opt.zero_grad()
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update()
            tot += loss.item()
        print(f"  Ep{ep+1} loss={tot/len(dl):.4f}")

    sp = os.path.join(LORA_DIR, cls)
    os.makedirs(sp, exist_ok=True)
    unet_l.save_pretrained(os.path.join(sp, "unet_lora"))
    te_l.save_pretrained(  os.path.join(sp, "te_lora"))
    print(f"  💾 Saved → {sp}")
    return unet_l, te_l


print("✅ LoRA training function ready.")

✅ LoRA training function ready.


---
## Step 5 — Run LoRA Fine-Tuning (all 5 classes)

Automatically **skips** classes with existing checkpoints (safe for resume after disconnect).

In [8]:
from huggingface_hub import login
login(token="YOUR_HF_TOKEN_HERE")

In [9]:
print("Loading Stable Diffusion 2.1 base model (~5GB first time)...")
tokenizer_sd    = CLIPTokenizer.from_pretrained(SD21_MODEL_ID, subfolder="tokenizer", cache_dir=SD_CACHE)
text_encoder_sd = CLIPTextModel.from_pretrained(SD21_MODEL_ID, subfolder="text_encoder", cache_dir=SD_CACHE).to(DEVICE)
vae_sd          = AutoencoderKL.from_pretrained(SD21_MODEL_ID, subfolder="vae", cache_dir=SD_CACHE).to(DEVICE)
unet_sd         = UNet2DConditionModel.from_pretrained(SD21_MODEL_ID, subfolder="unet", cache_dir=SD_CACHE).to(DEVICE)
scheduler_sd    = DDPMScheduler.from_pretrained(SD21_MODEL_ID, subfolder="scheduler", cache_dir=SD_CACHE)
print("✅ Base model loaded.\n")

t_start = time.time()
for i, cls in enumerate(CLASSES, 1):
    lora_path = os.path.join(LORA_DIR, cls)
    if os.path.exists(os.path.join(lora_path, "unet_lora")) and \
       os.path.exists(os.path.join(lora_path, "te_lora")):
        print(f"[{i}/{len(CLASSES)}] ⏭ Skipping {cls.replace('Tomato___','')} — checkpoint exists")
        continue
    print(f"\n[{i}/{len(CLASSES)}] Training...")
    train_sd21_lora(cls, tokenizer_sd, text_encoder_sd, vae_sd, unet_sd, scheduler_sd)
    # Reload fresh base model for next class (avoid LoRA stacking)
    del unet_sd, text_encoder_sd
    gc.collect(); torch.cuda.empty_cache()
    text_encoder_sd = CLIPTextModel.from_pretrained(SD21_MODEL_ID, subfolder="text_encoder", cache_dir=SD_CACHE).to(DEVICE)
    unet_sd         = UNet2DConditionModel.from_pretrained(SD21_MODEL_ID, subfolder="unet", cache_dir=SD_CACHE).to(DEVICE)

elapsed = time.time() - t_start
print(f"\n{'='*60}")
print(f"✅ LoRA fine-tuning complete! ({elapsed/60:.1f} min)")
print(f"{'='*60}")

Loading Stable Diffusion 2.1 base model (~5GB first time)...


Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

✅ Base model loaded.

[1/5] ⏭ Skipping Bacterial_spot — checkpoint exists
[2/5] ⏭ Skipping Early_blight — checkpoint exists
[3/5] ⏭ Skipping Late_blight — checkpoint exists
[4/5] ⏭ Skipping Septoria_leaf_spot — checkpoint exists
[5/5] ⏭ Skipping healthy — checkpoint exists

✅ LoRA fine-tuning complete! (0.0 min)


---
## Step 6 — Generate 2,671 Images Per Class

Settings: DPM-Solver++, 25 steps, guidance=7.5, 512×512, batch=4

Also copies all **real images** into `FID_REAL` for evaluation.

**Resume-safe:** skips any class that already has ≥2,671 images.

In [10]:

# Free training models from VRAM first
try:
    del unet_sd, text_encoder_sd, vae_sd, tokenizer_sd, scheduler_sd
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()


def generate_sd21_full(cls, target=TARGET_PER_CLASS):
    """Generate `target` synthetic images for one class using LoRA-adapted SD2.1."""
    dst = os.path.join(AUG_ROOT, cls)
    src = CLASS_FOLDER_MAP[cls]

    # Copy ALL real images into FID_REAL for this class
    real_dst = os.path.join(FID_REAL, cls)
    copied = 0
    for f in os.listdir(src):
        if f.lower().endswith(VALID_EXT):
            fid_p = os.path.join(real_dst, f)
            if not os.path.exists(fid_p):
                shutil.copy(os.path.join(src, f), fid_p)
                copied += 1
    if copied > 0:
        print(f"  📋 Copied {copied} real images → {real_dst}")

    existing = [f for f in os.listdir(dst) if f.lower().endswith(VALID_EXT)]
    needed   = target - len(existing)
    if needed <= 0:
        print(f"  ✅ {cls.replace('Tomato___',''):30s}: {len(existing)} images — already complete")
        # Make sure FID_FAKE also has them
        fake_dst = os.path.join(FID_FAKE, cls)
        for f in os.listdir(dst):
            if f.lower().endswith(VALID_EXT):
                fp = os.path.join(fake_dst, f)
                if not os.path.exists(fp):
                    shutil.copy(os.path.join(dst, f), fp)
        return

    print(f"\n🎨 Generating {needed} images: {cls.replace('Tomato___','')}")
    print(f"   (already have {len(existing)}, target {target})")

    pipe = StableDiffusionPipeline.from_pretrained(
        SD21_MODEL_ID, torch_dtype=torch.float16,
        use_safetensors=True, cache_dir=SD_CACHE).to(DEVICE)
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

    lora_path = os.path.join(LORA_DIR, cls)
    pipe.unet         = PeftModel.from_pretrained(pipe.unet, os.path.join(lora_path, "unet_lora"))
    pipe.text_encoder = PeftModel.from_pretrained(pipe.text_encoder, os.path.join(lora_path, "te_lora"))

    try:
        pipe.enable_xformers_memory_efficient_attention()
    except Exception:
        print("   ⚠ xformers not available, using default attention")
    pipe.enable_vae_slicing()

    prompt = DISEASE_PROMPTS[cls]
    gen = len(existing)  # start index from existing count
    with tqdm(total=needed, desc=f"  {cls.replace('Tomato___','')}") as bar:
        while gen < target:
            bs    = min(4, target - gen)
            seeds = [np.random.randint(0, 2**32 - 1) for _ in range(bs)]
            with torch.autocast("cuda"):
                imgs = pipe(
                    prompt=[prompt]*bs,
                    negative_prompt=[NEGATIVE_PROMPT]*bs,
                    num_inference_steps=25,
                    guidance_scale=7.5, height=512, width=512,
                    generator=[torch.Generator(DEVICE).manual_seed(int(s)) for s in seeds],
                ).images
            for img in imgs:
                fname = f"sd21_{gen:05d}.jpg"
                img.save(os.path.join(dst, fname), quality=95)
                img.save(os.path.join(FID_FAKE, cls, fname), quality=95)
                gen += 1; bar.update(1)
                if gen >= target: break

    del pipe; gc.collect(); torch.cuda.empty_cache()
    print(f"  ✅ Done — {gen} images in {dst}")


t_start = time.time()
for i, cls in enumerate(CLASSES, 1):
    print(f"\n[{i}/{len(CLASSES)}]")
    generate_sd21_full(cls)

elapsed = time.time() - t_start
print(f"\n{'='*60}")
print(f"✅ Generation complete! {TARGET_PER_CLASS} × {len(CLASSES)} = {TARGET_PER_CLASS*len(CLASSES)} images")
print(f"   Time: {elapsed/60:.1f} min")
print(f"   Output: {AUG_ROOT}")
print(f"{'='*60}")


[1/5]
  ✅ Bacterial_spot                : 2671 images — already complete

[2/5]
  ✅ Early_blight                  : 2671 images — already complete

[3/5]
  ✅ Late_blight                   : 2671 images — already complete

[4/5]
  ✅ Septoria_leaf_spot            : 2671 images — already complete

[5/5]

🎨 Generating 535 images: healthy
   (already have 2136, target 2671)


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

  healthy:   0%|          | 0/535 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

...

  0%|          | 0/25 [00:00<?, ?it/s]

  healthy:  99%|█████████▉| 529/535 [42:32<00:28,  4.81s/it]

  0%|          | 0/25 [00:00<?, ?it/s]

  healthy: 100%|██████████| 535/535 [42:47<00:00,  4.80s/it]


  ✅ Done — 2671 images in /content/drive/MyDrive/Tomato_SD21_Output/augmented_v2/Tomato___healthy

✅ Generation complete! 2671 × 5 = 13355 images
   Time: 43.4 min
   Output: /content/drive/MyDrive/Tomato_SD21_Output/augmented_v2


---
## Step 7 — Evaluation Functions (FID / KID / IS / Precision / Recall)

In [11]:
class InceptionV3Features(nn.Module):
    def __init__(self, output_blocks=("pool3",)):
        super().__init__()
        inception = models.inception_v3(
            weights=models.Inception_V3_Weights.DEFAULT, transform_input=False)
        self.Conv2d_1a_3x3 = inception.Conv2d_1a_3x3
        self.Conv2d_2a_3x3 = inception.Conv2d_2a_3x3
        self.Conv2d_2b_3x3 = inception.Conv2d_2b_3x3
        self.maxpool1      = nn.MaxPool2d(3, stride=2)
        self.Conv2d_3b_1x1 = inception.Conv2d_3b_1x1
        self.Conv2d_4a_3x3 = inception.Conv2d_4a_3x3
        self.maxpool2      = nn.MaxPool2d(3, stride=2)
        self.Mixed_5b = inception.Mixed_5b; self.Mixed_5c = inception.Mixed_5c
        self.Mixed_5d = inception.Mixed_5d; self.Mixed_6a = inception.Mixed_6a
        self.Mixed_6b = inception.Mixed_6b; self.Mixed_6c = inception.Mixed_6c
        self.Mixed_6d = inception.Mixed_6d; self.Mixed_6e = inception.Mixed_6e
        self.Mixed_7a = inception.Mixed_7a; self.Mixed_7b = inception.Mixed_7b
        self.Mixed_7c = inception.Mixed_7c
        self.avgpool  = nn.AdaptiveAvgPool2d((1, 1))
        self.fc       = inception.fc
        self.output_blocks = set(output_blocks)

    def forward(self, x):
        if x.shape[2] != 299 or x.shape[3] != 299:
            x = F.interpolate(x, size=(299, 299), mode="bilinear", align_corners=False)
        x = self.Conv2d_1a_3x3(x); x = self.Conv2d_2a_3x3(x); x = self.Conv2d_2b_3x3(x)
        x = self.maxpool1(x); x = self.Conv2d_3b_1x1(x); x = self.Conv2d_4a_3x3(x)
        x = self.maxpool2(x)
        x = self.Mixed_5b(x); x = self.Mixed_5c(x); x = self.Mixed_5d(x)
        x = self.Mixed_6a(x); x = self.Mixed_6b(x); x = self.Mixed_6c(x)
        x = self.Mixed_6d(x); x = self.Mixed_6e(x)
        x = self.Mixed_7a(x); x = self.Mixed_7b(x); x = self.Mixed_7c(x)
        pool3 = self.avgpool(x).flatten(1)
        if "pool3" in self.output_blocks and "logits" not in self.output_blocks:
            return {"pool3": pool3}
        logits = self.fc(pool3)
        probs  = F.softmax(logits, dim=1)
        return {"pool3": pool3, "probs": probs}

_inception_model = None
def get_inception(device, need_logits=False):
    global _inception_model
    if _inception_model is None:
        blocks = ("pool3", "logits") if need_logits else ("pool3",)
        _inception_model = InceptionV3Features(output_blocks=blocks).to(device)
        _inception_model.eval()
    return _inception_model

_to_tensor = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5]),
])

class _FolderDS(Dataset):
    def __init__(self, folder, tfm=_to_tensor):
        self.paths = [os.path.join(folder, f) for f in os.listdir(folder)
                      if f.lower().endswith((".jpg",".jpeg",".png"))]
        self.tfm = tfm
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        return self.tfm(Image.open(self.paths[i]).convert("RGB"))

def _extract_features(folder, inception_model, device, batch_size=32):
    loader = DataLoader(_FolderDS(folder), batch_size=batch_size, num_workers=2)
    feats = []
    with torch.no_grad():
        for batch in loader:
            out = inception_model(batch.to(device))
            feats.append(out["pool3"].cpu().numpy())
    return np.concatenate(feats, axis=0)

def _extract_probs(folder, inception_model, device, batch_size=32):
    loader = DataLoader(_FolderDS(folder), batch_size=batch_size, num_workers=2)
    probs = []
    with torch.no_grad():
        for batch in loader:
            out = inception_model(batch.to(device))
            probs.append(out["probs"].cpu().numpy())
    return np.concatenate(probs, axis=0)

def compute_fid(real_folder, fake_folder, inception, device):
    r = _extract_features(real_folder, inception, device)
    f = _extract_features(fake_folder, inception, device)
    if len(r) < 2 or len(f) < 2: return None
    mu_r, sigma_r = r.mean(0), np.cov(r, rowvar=False)
    mu_f, sigma_f = f.mean(0), np.cov(f, rowvar=False)
    diff = mu_r - mu_f
    covmean, _ = linalg.sqrtm(sigma_r.dot(sigma_f), disp=False)
    if np.iscomplexobj(covmean): covmean = covmean.real
    return float(diff.dot(diff) + np.trace(sigma_r) + np.trace(sigma_f) - 2*np.trace(covmean))

def _poly_mmd(X, Y, degree=3, gamma=None, coef0=1.0):
    if gamma is None: gamma = 1.0 / X.shape[1]
    XX = (gamma*X@X.T + coef0)**degree; YY = (gamma*Y@Y.T + coef0)**degree; XY = (gamma*X@Y.T + coef0)**degree
    n, m = X.shape[0], Y.shape[0]
    return float((XX.sum()-np.trace(XX))/(n*(n-1)) + (YY.sum()-np.trace(YY))/(m*(m-1)) - 2*XY.mean())

def compute_kid(real_folder, fake_folder, inception, device, n_subsets=100, subset_size=50):
    r = _extract_features(real_folder, inception, device)
    f = _extract_features(fake_folder, inception, device)
    if len(r) < subset_size or len(f) < subset_size: return None, None
    r = r / (np.linalg.norm(r, axis=1, keepdims=True)+1e-10)
    f = f / (np.linalg.norm(f, axis=1, keepdims=True)+1e-10)
    mmds = []
    rng = np.random.default_rng(42)
    for _ in range(n_subsets):
        ri = rng.choice(len(r), subset_size, replace=False)
        fi = rng.choice(len(f), subset_size, replace=False)
        mmds.append(_poly_mmd(r[ri], f[fi]))
    return float(np.mean(mmds)), float(np.std(mmds))

def compute_is(fake_folder, inception, device, splits=10):
    probs = _extract_probs(fake_folder, inception, device)
    n = probs.shape[0]; split_size = n // splits
    scores = []
    for k in range(splits):
        part = probs[k*split_size:(k+1)*split_size]
        py = part.mean(axis=0)
        kl = entropy(part.T, np.tile(py, (part.shape[0], 1)).T)
        scores.append(np.exp(kl.mean()))
    return float(np.mean(scores)), float(np.std(scores))

def compute_precision_recall(real_folder, fake_folder, inception, device, k=3):
    r = _extract_features(real_folder, inception, device)
    f = _extract_features(fake_folder, inception, device)
    if len(r) < 4 or len(f) < 4: return None, None
    def _nn_radii(X, k):
        nbrs = NearestNeighbors(n_neighbors=k+1).fit(X)
        dists, _ = nbrs.kneighbors(X)
        return dists[:, -1]
    real_radii = _nn_radii(r, k); fake_radii = _nn_radii(f, k)
    nbrs_real = NearestNeighbors(n_neighbors=1).fit(r)
    dist_f2r, _ = nbrs_real.kneighbors(f)
    precision = float((dist_f2r[:,0] <= real_radii.mean()).mean())
    nbrs_fake = NearestNeighbors(n_neighbors=1).fit(f)
    dist_r2f, _ = nbrs_fake.kneighbors(r)
    recall = float((dist_r2f[:,0] <= fake_radii.mean()).mean())
    return precision, recall

print("✅ Evaluation functions ready.")

✅ Evaluation functions ready.


---
## Step 8 — Run Full Evaluation

In [12]:
def run_evaluation():
    print("Loading InceptionV3...")
    inception = get_inception(DEVICE, need_logits=True)
    results = {}
    for cls in CLASSES:
        short = cls.replace("Tomato___", "").replace("_", " ")
        real_dir = os.path.join(FID_REAL, cls)
        fake_dir = os.path.join(FID_FAKE, cls)
        n_real = len([f for f in os.listdir(real_dir) if f.lower().endswith(VALID_EXT)])
        n_fake = len([f for f in os.listdir(fake_dir) if f.lower().endswith(VALID_EXT)])
        print(f"\n{'─'*50}")
        print(f"  {short} ({n_real} real / {n_fake} fake)")
        print(f"{'─'*50}")
        rec = {"n_real": n_real, "n_fake": n_fake}
        if n_real < 4 or n_fake < 4:
            print("  ⚠ Not enough images — skipping")
            results[cls] = rec; continue

        print("  FID ...", end=" ")
        rec["FID"] = compute_fid(real_dir, fake_dir, inception, DEVICE)
        print(f"{rec['FID']:.3f}")

        print("  KID ...", end=" ")
        kid_mean, kid_std = compute_kid(real_dir, fake_dir, inception, DEVICE,
                                         subset_size=min(50, n_real//2, n_fake//2))
        rec["KID_mean"], rec["KID_std"] = kid_mean, kid_std
        print(f"{kid_mean:.5f} +/- {kid_std:.5f}")

        print("  IS ...", end=" ")
        is_mean, is_std = compute_is(fake_dir, inception, DEVICE, splits=min(10, max(1, n_fake//10)))
        rec["IS_mean"], rec["IS_std"] = is_mean, is_std
        print(f"{is_mean:.3f} +/- {is_std:.3f}")

        print("  Precision/Recall ...", end=" ")
        prec, rc = compute_precision_recall(real_dir, fake_dir, inception, DEVICE, k=3)
        rec["Precision"], rec["Recall"] = prec, rc
        print(f"P={prec:.4f}  R={rc:.4f}")

        results[cls] = rec
        gc.collect(); torch.cuda.empty_cache()
    return results

results = run_evaluation()

Loading InceptionV3...
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 162MB/s] 



──────────────────────────────────────────────────
  Bacterial spot (331 real / 2671 fake)
──────────────────────────────────────────────────
  FID ... 159.649
  KID ... 0.00024 +/- 0.00004
  IS ... 1.772 +/- 0.083
  Precision/Recall ... P=0.0000  R=0.0030

──────────────────────────────────────────────────
  Early blight (330 real / 2671 fake)
──────────────────────────────────────────────────
  FID ... 104.118
  KID ... 0.00012 +/- 0.00002
  IS ... 2.369 +/- 0.094
  Precision/Recall ... P=0.0041  R=0.0303

──────────────────────────────────────────────────
  Late blight (329 real / 2671 fake)
──────────────────────────────────────────────────
  FID ... 125.328
  KID ... 0.00011 +/- 0.00002
  IS ... 2.792 +/- 0.102
  Precision/Recall ... P=0.1011  R=0.0851

──────────────────────────────────────────────────
  Septoria leaf spot (302 real / 2671 fake)
──────────────────────────────────────────────────
  FID ... 128.224
  KID ... 0.00019 +/- 0.00003
  IS ... 2.137 +/- 0.071
  Precision

---
## Step 9 — Save Results (CSV + XLSX)

In [13]:
def save_results(results):
    rows = []
    for cls in CLASSES:
        r = results.get(cls, {})
        rows.append({
            "Disease class": cls.replace("Tomato___", "").replace("_", " "),
            "n_real": r.get("n_real", "N/A"), "n_fake": r.get("n_fake", "N/A"),
            "FID": r.get("FID", "N/A"),
            "KID": r.get("KID_mean", "N/A"), "KID_std": r.get("KID_std", "N/A"),
            "IS": r.get("IS_mean", "N/A"), "IS_std": r.get("IS_std", "N/A"),
            "Precision": r.get("Precision", "N/A"), "Recall": r.get("Recall", "N/A"),
        })
    csv_path = os.path.join(OUTPUT_ROOT, "generation_quality_full_2671.csv")
    with open(csv_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader(); w.writerows(rows)
    print(f"📄 Saved: {csv_path}")
    try:
        import pandas as pd
        xlsx_path = os.path.join(OUTPUT_ROOT, "generation_quality_full_2671.xlsx")
        df = pd.DataFrame(rows)
        df.to_excel(xlsx_path, index=False)
        print(f"📊 Saved: {xlsx_path}")
        return df
    except ImportError:
        print("pandas/openpyxl not installed — only CSV saved.")
        return None

df_results = save_results(results)
df_results

📄 Saved: /content/drive/MyDrive/Tomato_SD21_Output/generation_quality_full_2671.csv
📊 Saved: /content/drive/MyDrive/Tomato_SD21_Output/generation_quality_full_2671.xlsx


,Disease class,n_real,n_fake,FID,KID,KID_std,IS,IS_std,Precision,Recall
0,Bacterial spot,331,2671,159.648953,0.000238,0.000036,1.771831,0.082731,0.000000,0.003021
1,Early blight,330,2671,104.118464,0.000121,0.000019,2.369466,0.093733,0.004118,0.030303
2,Late blight,329,2671,125.327587,0.000111,0.000016,2.791935,0.102059,0.101086,0.085106
3,Septoria leaf spot,302,2671,128.223981,0.000186,0.000026,2.137242,0.070584,0.044927,0.026490
4,healthy,330,2671,65.791073,0.000099,0.000010,1.429991,0.025738,0.019094,0.054545


---
## Step 10 — Create ZIP of All Generated Images

Creates a single ZIP file with the folder structure:
```
augmented_v2_all_classes.zip
├── Tomato___Bacterial_spot/      (2,671 images)
├── Tomato___Early_blight/        (2,671 images)
├── Tomato___Late_blight/         (2,671 images)
├── Tomato___Septoria_leaf_spot/  (2,671 images)
└── Tomato___healthy/             (2,671 images)
```

In [14]:
print("📦 Creating ZIP archive...")
print(f"   Source: {AUG_ROOT}")
print(f"   Target: {ZIP_PATH}")

total_files = 0
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_STORED) as zf:
    for cls in CLASSES:
        cls_dir = os.path.join(AUG_ROOT, cls)
        imgs = [f for f in sorted(os.listdir(cls_dir)) if f.lower().endswith(VALID_EXT)]
        print(f"   Adding {cls}: {len(imgs)} images...", end=" ")
        for img_name in imgs:
            full_path = os.path.join(cls_dir, img_name)
            arcname = os.path.join(cls, img_name)  # relative path inside zip
            zf.write(full_path, arcname)
            total_files += 1
        print("✅")

zip_size_gb = os.path.getsize(ZIP_PATH) / (1024**3)
print(f"\n{'='*60}")
print(f"✅ ZIP created successfully!")
print(f"   📦 {ZIP_PATH}")
print(f"   📊 {total_files} images, {zip_size_gb:.2f} GB")
print(f"{'='*60}")

📦 Creating ZIP archive...
   Source: /content/drive/MyDrive/Tomato_SD21_Output/augmented_v2
   Target: /content/drive/MyDrive/Tomato_SD21_Output/augmented_v2_all_classes.zip
   Adding Tomato___Bacterial_spot: 2671 images... ✅
   Adding Tomato___Early_blight: 2671 images... ✅
   Adding Tomato___Late_blight: 2671 images... ✅
   Adding Tomato___Septoria_leaf_spot: 2671 images... ✅
   Adding Tomato___healthy: 2671 images... ✅

✅ ZIP created successfully!
   📦 /content/drive/MyDrive/Tomato_SD21_Output/augmented_v2_all_classes.zip
   📊 13355 images, 1.05 GB


---
## Step 11 — Download ZIP (optional)

Run this cell to trigger a browser download of the ZIP file.
You can also find it directly in your Google Drive at:
`My Drive/Tomato_SD21_Output/augmented_v2_all_classes.zip`

In [17]:

from google.colab import files
files.download(ZIP_PATH)

print(f"             My Drive/Tomato_SD21_Output/augmented_v2_all_classes.zip")
print(f"\n📁 All outputs are saved in: {OUTPUT_ROOT}")
print(f"   ├── augmented_v2/          — generated images (organized by class)")
print(f"   ├── sd21_lora/             — LoRA checkpoints (reusable)")
print(f"   ├── fid_real_v2/           — real images (for evaluation)")
print(f"   ├── fid_fake_v2/           — fake images (for evaluation)")
print(f"   ├── generation_quality_full_2671.csv")
print(f"   ├── generation_quality_full_2671.xlsx")
print(f"   └── augmented_v2_all_classes.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

             My Drive/Tomato_SD21_Output/augmented_v2_all_classes.zip

📁 All outputs are saved in: /content/drive/MyDrive/Tomato_SD21_Output
   ├── augmented_v2/          — generated images (organized by class)
   ├── sd21_lora/             — LoRA checkpoints (reusable)
   ├── fid_real_v2/           — real images (for evaluation)
   ├── fid_fake_v2/           — fake images (for evaluation)
   ├── generation_quality_full_2671.csv
   ├── generation_quality_full_2671.xlsx
   └── augmented_v2_all_classes.zip
